In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import tensorflow as tf

# 1. Define Data Paths and Parameters{ska}
images_path = '/kaggle/input/brain-ds/images'  # Path to image folder
masks_path = '/kaggle/input/brain-ds/masks'    # Path to mask folder
img_size = 128  # Reduced image size for faster processing
batch_size = 32
epochs = 10

# 2. Load and Preprocess Data{ska}
def load_data(images_path, masks_path, img_size):
    images = []
    masks = []
    masked_images = [] # To store {ska}masked images
    image_files = sorted(os.listdir(images_path))
    mask_files = sorted(os.listdir(masks_path))

    for i in range(len(image_files)):
    
        # Load and resize image
        img = load_img(os.path.join(images_path, image_files[i]), target_size=(img_size, img_size))
        img_array = img_to_array(img) / 255.0  # Normalize {ska}pixel values
        images.append(img_array)

        # Load and resize mask (grayscale)
        mask = load_img(os.path.join(masks_path, mask_files[i]), target_size=(img_size, img_size), color_mode='grayscale')
        mask_array = img_to_array(mask) / 255.0  # N{ska}ormalize
        masks.append(mask_array)

        # Create masked image
        masked_img = img_array * mask_array  # Element-wise multiplication
        masked_images.append(masked_img)

    return np.array(images), np.array(masks), np.array(masked_images) # Return also the masked images

images, masks, masked_images = load_data(images_path, masks_path, img_size) # Load the masked images
#Split the data
X_train, X_test, y_train, y_test = train_test_split(np.concatenate((images, masked_images), axis=-1), masks, test_size=0.2, random_state=42)


# 3. Define the CNN Model (Modified U-Net)
def create_model(img_size):
    inputs = Input((img_size, img_size, 6)) # Input now has 6 channels (3 from image, 3 from masked image)

    # Encoder
    conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    pool1 = MaxPooling2D((2, 2))(conv1)
    conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)
    pool2 = MaxPooling2D((2, 2))(conv2)
    conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool2)

    # Decoder
    up1 = UpSampling2D((2, 2))(conv3)
    concat1 = Concatenate()([conv2, up1]) # concatenate
    conv4 = Conv2D(64, (3, 3), activation='relu', padding='same')(concat1)
    up2 = UpSampling2D((2, 2))(conv4)
    concat2 = Concatenate()([conv1, up2])
    conv5 = Conv2D(1, (1, 1), activation='sigmoid')(concat2) # Output layer for mask

    model = Model(inputs=inputs, outputs=conv5)
    model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# if tf.config.list_physical_devices('GPU'):
#     # If GPU is a{ska}vailable, use it
#     print("GPU is available. Training on GPU.")
#     print(f"TensorFlow version: {tf.__version__}")
#     model = create_model(img_size)
#     model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(X_test,y_test))
# else:
#     # If no GPU is avai{ska}lable, use CPU
#     print("GPU is not available. Training on CPU.")
#     print(f"TensorFlow version: {tf.__version__}")
#     with tf.device('/CPU:0'):
#         model = create_model(img_size)
#         model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(X_test,y_test))

# # 5. Save the Mo{ska}del (Optional)
# model.save('tumor_segmentation_model.h5')
# print("Evaluating the model on the test set...")
# results = model.evaluate(X_test, y_test, batch_size=batch_size)
# print(f"Test Loss: {results[0]}, Test Accuracy: {results[1]}, Test Mean IoU: {results[2]}")



In [ ]:
# import streamlit as st  # Streamlit for UI
from PIL import Image  # For image processing within Streamlit
from io import BytesIO #for converting the image
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

def main():
        dice_scores = []
        precision_scores = []
        recall_scores = []
        iou_scores = []
###############
        img = load_img("/kaggle/input/brain-ds/images/100.png", color_mode='grayscale', target_size=(128, 128))
        img_array = img_to_array(img) / 255.0 # Normalize to [0, 1]

        # Assuming the masked image is{ska} also an RGB image
        masked_img = load_img("/kaggle/input/brain-ds/masks/100.png", color_mode='grayscale', target_size=(128, 128))
        masked_img_array = img_to_array(masked_img) / 255.0 # Normalize to [0, 1]
        
        # Concatenate the image and masked image arrays along the channel axis
        # Resulting shape will be (target_size, target_size, 6){ska}
        combined_input = np.concatenate([img_array, masked_img_array], axis=0)

        # Add a batch dimension (for a single image prediction)
        # Resulting shape will be (1, target_size, target_size, 6)
        combined_input = np.expand_dims(combined_input, axis=0)
###########
        model= tf.keras.models.load_model('/kaggle/input/tumor/tensorflow2/default/1/tumor_segmentation_model.h5')
        model.predict(combined_input)
       
       
       
        #     # Calculate metrics for the batch
        dice_scores.append(dice_coef(segmented_mask, preds))
        precision_scores.append(precision(segmented_mask, preds))
        recall_scores.append(recall(segmented_mask, preds))
        iou_scores.append(iou(segmented_mask, preds))


        mean_dice = tf.reduce_mean(tf.stack(dice_scores)).numpy()
        mean_precision = tf.reduce_mean(tf.stack(precision_scores)).numpy()
        mean_recall = tf.reduce_mean(tf.stack(recall_scores)).numpy()
        mean_iou = tf.reduce_mean(tf.stack(iou_scores)).numpy()


        print(f"Dice Coefficient: {mean_dice:.4f}")
        print(f"Precision: {mean_precision:.4f}")
        print(f"Recall: {mean_recall:.4f}")
        print(f"IoU: {mean_iou:.4f}")
        # # 7.5. Display Results
        # st.subheader("Uploaded Image")
        # st.image(image, caption="Original MRI Image", use_column_width=True)

        # st.subheader("Segmented Tumor")
        segmented_image = Image.fromarray((segmented_mask * 255).squeeze(),
                                         mode='L')  # convert back to image
        # st.image(segmented_image, caption="Predicted Tumor Mask", use_column_width=True)

        # 7.6. Tumor/No Tumor Prediction
        tumor_present = np.any(segmented_mask)  # Check if any non-zero pixel in mask
        # if tumor_present:
        #     st.warning("The image is predicted to contain a tumor.")
        # else:
        #     st.success("The image is predicted to be tumor-free.")

        # 7.7. Downlo{ska}ad Segmentation Result
        buffered = BytesIO()
        segmented_image.save(buffered, format="PNG")
        # st.download_button(
        #     label="Download Segmented Mask",
        #     data=buffered.getvalue(),
        #     file_name="segmented_mask.png",
        #     mime="image/png",
        # )
        print(f"Dice Coefficient: {mean_dice:.4f}")
        print(f"Precision: {mean_precision:.4f}")
        print(f"Recall: {mean_recall:.4f}")
        print(f"IoU: {mean_iou:.4f}")
def load_model(model_path):
         # In a real app, you would load your traine{ska}d model
         # For this example, we'll just create a new model if the file doesn't exist
         if tf.io.gfile.exists(model_path):
             return tf.keras.models.load_model(model_path)
         else:
             st.warning(f"Model file not found at {model_path}.")



if __name__ == "__main__":
    main()

In [ ]:
!python -v

In [ ]:
import streamlit as st  # Streamlit for UI
from PIL import Image  # For image processing{ska} within Streamlit
from io import BytesIO #for converting the {ska}image



def train_and_evaluate_model(X_train, y_train, X_test, y_test, img_size):
    # Check for GPU availability
    if tf.config.list_physical_devices('GPU'):
        # If GPU is available, use it
        print("GPU is available. Training on GPU.")
        print(f"TensorFlow version: {tf.__version__}")
        model = create_model(img_size)
        model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs,
                          validation_data=(X_test, y_test))
    else:
        # If no GPU is available, use CPU
        print("GPU is not available. Training on CPU.")
        print(f"TensorFlow version: {tf.__version__}")
        with tf.device('/CPU:0'):
            model = create_model(img_size)
            model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs,
                              validation_data=(X_test, y_test))

    # 5. Save the Model (Optional)
    model.save(model_path)
    print(f"Model saved to {model_path}")

    # 6. Evaluate the model on the t{ska}est set
    print("Evaluating the model on the test set...")
    results = model.evaluate(X_test, y_test, batch_size=batch_size)
    print(f"Test Loss: {results[0]:.4f}, Test Accuracy: {results[1]:.4f}, Test Mean IoU: {results[2]:.4f}")
    return model, results  # Return the trained model and evaluation results



# 7. Streamlit App for{ska} User Interaction
def main():
    st.title("Tumor Segmentation Web App")

    # 7.1. File Upload
    uploaded_file = st.file_uploader("Upload an MRI image...", type=["png", "jpg", "jpeg"])

    if uploaded_file is not None:
        # 7.2. Read and preprocess the uploaded image
        image = Image.open(uploaded_file)
        img = image.resize((img_size, img_size))
        img_array = np.array(img) / 255.0
        input_img = np.expand_dims(img_array, axis=0)  # Add batch dimension

        # 7.3. Load the trained model
        model = load_model(model_path)  # Load the pre-trained model

        # 7.4. Predict the seg{ska}mentation mask
        prediction = model.predict(input_img)
        segmented_mask = (prediction[0] > 0.5).astype(np.uint8)  # Threshold the prediction

        # 7.5. Display Re{ska}sults
        st.subheader("Uploaded Image")
        st.image(image, caption="Original MRI Image", use_column_width=True)

        st.subheader("Segmented Tumor")
        segmented_image = Image.fromarray((segmented_mask * 255).squeeze(),
                                         mode='L')  # convert back to image
        st.image(segmented_image, caption="Predicted Tumor Mask", use_column_width=True)

        # 7.6. Tumor/No T{ska}umor Prediction
        tumor_present = np.any(segmented_mask)  # Check if any non-zero pixel in mask
        if tumor_present:
            st.warning("The image is predicted to contain a tumor.")
        else:
            st.success("The image is predicted to be tumor-free.")

        # 7.7. Download Segmentation Result
        buffered = BytesIO()
        segmented_image.save(buffered, format="PNG")
        st.download_button(
            label="Download Segmented Mask",
            data=buffered.getvalue(),
            file_name="segmented_mask.png",
            mime="image/png",
        )

    # 7.8 Train and Evaluate Model Option
    if st.sidebar.checkbox("Train and Evaluate Model"):
        st.subheader("Training and Evaluation")
        # Train and evaluate the model
        trained_model, evaluation_results = train_and_evaluate_model(X_train, y_train, X_test, y_test, img_size)
        st.write(f"Test Loss: {evaluation_results[0]:.4f}")
        st.write(f"Test Accuracy: {evaluation_results[1]:.4f}")
        st.write(f"Test Mean IoU: {evaluation_results[2]:.4f}")
        st.success("Model training and evaluation complete!")

if __name__ == "__main__":
    main()

In [ ]:
!pip3 install streamlit

In [ ]:
!streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py